**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2019"], check=True)
    os.chdir("/content/encore/entries/tingley-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Tingley and Buzsaki (2019) from DANDI:000218: LFP analyses

Tingley D. and Buzsaki G. (2019). Routing of hippocampal ripples to subcortical structures via the lateral septum. Neuron 105, 138-149. doi:10.1016/j.neuron.2019.10.012.

Dataset: DANDI:000218, version 0.261110.1911 (doi:10.48324/dandi.000218/0.261110.1911), streamed from the archive.

Version. The paper cites no version of the dandiset (it predates it). The most recent published version, 0.261110.1911, is pinned: its files were re-converted with trial labels sorted with their rows, the lab's ripple and LS HFO detections, sleep states for every session with them, units with shank and region, and electrode flags, all of which the claims use; the earlier versions (0.220131.1608, 0.250624.0439) lack the event tables and sleep states, so they match the claims file's description less well. The LFP is the earlier version's data, stored again in channel order (release notes).

Claims file: `claims/tingley-2019.yaml`, written from the accepted manuscript in PubMed Central (PMC6952543) with its supplement. The paper's full text and supplement were read here.

This notebook tests the claims that need the LFP: phase locking of LS spikes, locking to the CA1 ripple band, event times at the maximum NSS, CA1-LS phase coherence, and HFO power by recording site.

Claims and where they are tested (all notebooks of this entry):

| Claim | Where tested |
|---|---|
| dataset-composition | main notebook (counts); LFP notebook (LS neurons with more than 50 spikes) |
| fig1c-hfo-each-animal, fig1c-post-burst-suppression, fig1c-event-count, figs1-participation | main notebook |
| fig1d-phase-locking | LFP notebook; own-shank notebook (shanks outside the detection channel's storage block) |
| fig1f-shorter-than-ripples, fig1g-slower-than-ripples, fig1h-no-amplitude-frequency-correlation | main notebook |
| fig2-confined-to-ls | LFP notebook |
| fig2ef-intermediate-ls | blocked: metadata_insufficient (no relative depth of each recording within LS) |
| fig3-travelling-waves | blocked: metadata_insufficient (no electrode positions) |
| fig4b-pre-post-unchanged, fig4b-hfo-above-behav, fig4-sessions-few-change, fig4c-temporal-bias, fig4d-compression | main notebook |
| figs3d-hpc-compression-post, figs2-post-rate-increase, figs2c-no-ls-reactivation | main notebook |
| fig5b-nrem-and-rates, fig5e-state-coupling, fig5f-ripple-triggered-firing | main notebook |
| fig5d-lag | main notebook (deposited peak times); LFP notebook (maximum-NSS times) |
| fig5f-no-phase-coherence, ls-spikes-not-locked-to-ca1-ripples | LFP notebook |
| fig6-optogenetic-hfos | blocked: data_missing (the two optogenetic rats are not in the dandiset) |
| fig7e-ca1-spikes, fig7e-state, fig7e-duration | Figure 7E notebook |
| fig7e-ca1-depth | blocked: metadata_insufficient (no site positions or per-site waveforms) |
| fig7e-replay, fig7e-seqnmf, figs5-seqnmf-variance | blocked: other (not attempted within the run's time) |
| fig7e-content-independent, fig7e-coupled-only | blocked: other (the content features, fig7e-replay and fig7e-seqnmf, were not attempted) |
| discussion-medial-septum | main notebook (part b); LFP notebook (part a) |
| discussion-cingulate-down-state | main notebook (reading a) |
| fig8-model | blocked: other (the model's data are elsewhere and it was not attempted within the run's time) |

## Setup
Every file of the pinned version is streamed; from each, the spikes, units, event tables, sleep states,
trials and electrode flags are read into memory. The LFP is not read in this notebook.

In [ ]:
import zlib, time, resource, collections, warnings
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import tingley2019 as T
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
KEY = "tingley-2019"
def seed_of(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
t0 = time.time()
A = T.list_assets()
S = T.load_all(A, workers=8)
print(f"{len(S)} files of dandiset {T.DANDISET} version {T.VERSION} loaded in {time.time()-t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Data check
What was loaded, by subject and session; the ranges and units of every variable used; recordings deposited
twice; and whether the spikes, the event tables, the sleep states and the trials share one clock.

In [ ]:
def ls_ev(s, fallback=False):
    """The lab's main LS HFO detection (LSRipples); with fallback, the earlier ls_ripples, then ripples_ls."""
    for k in (["LSRipples", "ls_ripples", "ripples_ls"] if fallback else ["LSRipples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
def ca1_ev(s, fallback=False):
    """The lab's CA1 ripples (CA1Ripples); with fallback, the hippocampal ripplesALL."""
    for k in (["CA1Ripples", "ripplesALL"] if fallback else ["CA1Ripples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
HPC = list(T.HPC_REGIONS)
rows = []
for s in S:
    reg = collections.Counter(s.unit_region)
    sl = {} if s.sleep is None else (s.sleep.stop_time - s.sleep.start_time).groupby(s.sleep.label).sum().to_dict()
    rows.append(dict(subject=s.subject, session=s.session[:30], hours=round(s.duration/3600, 2), lfp_channels=s.lfp_shape[1],
                     ls_units=reg.get("ls", 0), hpc_units=reg.get("hpc", 0), ca1_units=reg.get("ca1", 0), ca3_units=reg.get("ca3", 0),
                     other_units=sum(v for k, v in reg.items() if k not in ("ls", "hpc", "ca1", "ca3")),
                     other_regions=",".join(sorted(k if k else "''" for k in reg if k not in ("ls", "hpc", "ca1", "ca3"))),
                     tables=",".join(f"{k}:{len(v)}" for k, v in sorted(s.events.items())),
                     nrem_h=round(sl.get("NREM", 0)/3600, 2), wake_h=round(sl.get("WAKE", 0)/3600, 2), rem_h=round(sl.get("REM", 0)/3600, 2),
                     trials=0 if s.trials is None else len(s.trials),
                     ca3_flag=bool(s.electrodes.ca3_channel.any()), ca1_flag=bool(s.electrodes.ca1_channel.any())))
INV = pd.DataFrame(rows)
print(INV.to_string())
print("\nBy subject:")
print(INV.groupby("subject")[["ls_units", "hpc_units", "ca3_units", "other_units", "trials"]].sum()
      .join(INV.groupby("subject").size().rename("files")).to_string())
print("\nUnit regions over all files:", dict(collections.Counter(r for s in S for r in s.unit_region)))
print("Event tables (files holding each):", dict(collections.Counter(k for s in S for k in s.events)))

In [ ]:
# Ranges and units of the variables used.
for k in ["LSRipples", "CA1Ripples", "ripplesALL", "ls_ripples", "ripples_ls"]:
    d = pd.concat([s.events[k] for s in S if k in s.events])
    dur = (d.stop_time - d.start_time) * 1000
    inside = np.mean([((s.events[k].start_time >= 0) & (s.events[k].stop_time <= s.duration + 1)).mean() for s in S if k in s.events])
    pk = ((d.peak_time >= d.start_time) & (d.peak_time <= d.stop_time)).mean()
    fr = f"peak_frequency {d.peak_frequency.min():.1f} to {d.peak_frequency.max():.1f} (median {d.peak_frequency.median():.1f}, read as Hz)" if "peak_frequency" in d else "no peak_frequency"
    print(f"{k}: {len(d)} events; duration {dur.min():.1f} to {dur.max():.1f} ms (median {dur.median():.1f}); {fr}; "
          f"peak inside start-stop {pk:.3f}; events inside the recording {inside:.3f}")
allsp = [(s.spikes[i].min(), s.spikes[i].max(), s.duration) for s in S for i in range(len(s.spikes)) if len(s.spikes[i])]
print(f"Spike times: {sum(len(x) for s in S for x in s.spikes)} spikes in {sum(len(s.spikes) for s in S)} units; "
      f"min {min(a for a,_,_ in allsp):.4f} s; units with a spike after the LFP's end: {sum(b > d + 1 for _, b, d in allsp)}")
print("Sorted spike times in every unit:", all(np.all(np.diff(x) >= 0) for s in S for x in s.spikes))
sl = pd.concat([s.sleep.assign(dur=s.duration) for s in S if s.sleep is not None])
print(f"Sleep states: labels {sorted(sl.label.unique())}; bouts {len(sl)}; stop beyond the recording: {(sl.stop_time > sl.dur + 1).sum()}; "
      f"whole-second times: {np.mean(np.isclose(sl.start_time % 1, 0)):.3f}")
tr = pd.concat([s.trials.assign(dur=s.duration) for s in S if s.has_trials])
print(f"Trials: {len(tr)} in {sum(s.has_trials for s in S)} files; length {(tr.stop_time-tr.start_time).min():.2f} to "
      f"{(tr.stop_time-tr.start_time).max():.1f} s (median {(tr.stop_time-tr.start_time).median():.1f}); beyond the recording: {(tr.stop_time > tr.dur + 1).sum()}")

In [ ]:
# Recordings deposited twice: identical spike trains (unit spike counts and first spikes) or identical event tables.
fp_sp, fp_ev, dup = {}, {}, []
for i, s in enumerate(S):
    if len(s.spikes):
        key = tuple(sorted((len(x), round(float(x[0]), 4) if len(x) else 0) for x in s.spikes))
        if key in fp_sp: dup.append(("spikes", S[fp_sp[key]].session, s.session))
        fp_sp[key] = i
    for k, e in s.events.items():
        key = (len(e), tuple(np.round(e.start_time.to_numpy()[:20], 4)))
        if len(e) > 5 and key in fp_ev and fp_ev[key][0] != i:
            dup.append((f"events {k} / {fp_ev[key][1]}", S[fp_ev[key][0]].session, s.session))
        fp_ev[key] = (i, k)
# also: same subject, same LFP length
lens = collections.defaultdict(list)
for s in S: lens[(s.subject, s.lfp_shape[0])].append(s.session)
dup += [("same LFP length", *v) for v in lens.values() if len(v) > 1]
print("Duplicate candidates:", dup if dup else "none")
# Within one file, two event tables holding the same events (an LS table copied into another) are reported too.
same = []
for s in S:
    ks = sorted(s.events)
    for a in range(len(ks)):
        for b in range(a + 1, len(ks)):
            ea, eb = s.events[ks[a]].peak_time.to_numpy(), s.events[ks[b]].peak_time.to_numpy()
            if len(ea) and len(eb):
                j = np.clip(np.searchsorted(eb, ea), 1, len(eb) - 1)
                near = np.minimum(np.abs(eb[j] - ea), np.abs(eb[j - 1] - ea)) < 0.002
                if near.mean() > 0.9 and abs(len(ea) - len(eb)) < 0.1 * len(ea):
                    same.append((s.session[:25], ks[a], ks[b], len(ea), len(eb)))
print("Tables within a file with the same events:", same if same else "none")

In [ ]:
# One clock. (1) LS spikes around LS HFO peak times and hippocampal spikes around CA1 ripple peak times: the rate
# within +-10 ms over the rate 100-200 ms away, with the true times and with the times shifted by 1 s.
# (2) Trials against sleep scoring: the fraction of trial time scored WAKE, with the true states and with
# the states shifted by +-1800 s. A shared clock gives a ratio far above 1 at the true times only, and trials in WAKE.
def peri_ratio(s, units, times):
    c = sum(T.triggered_counts(s.spikes[u], times, 0.2, 0.01).sum(0) for u in units)
    lag = np.arange(-20, 21)
    return c[np.abs(lag) <= 1].mean() / max(c[(np.abs(lag) >= 10)].mean(), 1e-9)
clk = []
for s in S:
    ls = s.units_in(["ls"]); hp = s.units_in(HPC)
    e1, e2 = ls_ev(s), ca1_ev(s)
    r = dict(session=s.session[:25], subject=s.subject)
    if e1 is not None and len(ls):
        t = e1.peak_time.to_numpy(); r["ls_hfo"] = peri_ratio(s, ls, t); r["ls_hfo_shift1s"] = peri_ratio(s, ls, t + 1.0)
    if e2 is not None and len(hp):
        t = e2.peak_time.to_numpy(); r["ca1_rip"] = peri_ratio(s, hp, t); r["ca1_rip_shift1s"] = peri_ratio(s, hp, t + 1.0)
    if s.has_trials and s.sleep is not None:
        wk = s.state_intervals("WAKE"); tiv = s.trials[["start_time", "stop_time"]].to_numpy()
        r["trial_in_wake"] = T.total(T.intersect(tiv, wk)) / T.total(tiv)
        r["trial_in_wake_shift"] = np.mean([T.total(T.intersect(tiv, wk + d)) / T.total(tiv) for d in (-1800, 1800)])
    clk.append(r)
CLK = pd.DataFrame(clk)
print(CLK.describe().loc[["count", "min", "50%", "max"]].round(3).to_string())
print("Files where the LS HFO ratio is below 1.5:", CLK[CLK.ls_hfo < 1.5].session.tolist())
print("Files where the CA1 ripple ratio is below 1.5:", CLK[CLK.ca1_rip < 1.5].session.tolist())
print("Files where less than half the trial time is scored WAKE:", CLK[CLK.trial_in_wake < 0.5][["session", "trial_in_wake"]].values.tolist())

In [ ]:
from concurrent.futures import ThreadPoolExecutor
FS = T.FS_LFP
def nearest_lag(ls_t, ca_t):
    """LS time minus the nearest CA1 time (as in the main notebook, fig5d-lag)."""
    j = np.clip(np.searchsorted(ca_t, ls_t), 1, len(ca_t) - 1)
    a, b = ls_t - ca_t[j - 1], ls_t - ca_t[j]
    return np.where(np.abs(a) < np.abs(b), a, b)

## LFP analyses
One job per recording with LS units and the lab's LS HFO detection channel (the electrode marked `LSRipples` in
`ripple_detection_channel`). Each job reads the 16-channel storage block holding that channel (reading one channel of a
block costs as much as reading all of it), and, where the recording has CA1Ripples, the block of the CA1 detection channel.
From these: phase locking of LS spikes (fig1d-phase-locking, dataset-composition reading a), locking of LS spikes to the
CA1 channel (ls-spikes-not-locked-to-ca1-ripples), event times at the maximum NSS (fig5d-lag), and CA1-LS 120-180 Hz phase
coherence around CA1 ripples (fig5f-no-phase-coherence). Random draws (null shifts, jitters) come from a generator seeded
with the claim's seed and the recording's index.

In [ ]:
def det_row(s, name):
    r = np.flatnonzero(s.electrodes.ripple_detection_channel.str.contains(name).to_numpy())
    return int(r[0]) if len(r) else None
LFP_SESS = [s for s in S if det_row(s, "LSRipples") is not None and len(s.units_in(["ls"])) and ls_ev(s) is not None]
print(f"Recordings for the LFP jobs: {len(LFP_SESS)} ({collections.Counter(s.subject for s in LFP_SESS)}); with a CA1 detection channel and "
      f"CA1Ripples: {sum(det_row(s, 'CA1Ripples') is not None and ca1_ev(s) is not None for s in LFP_SESS)}")
print(f"LS units in them: {sum(len(s.units_in(['ls'])) for s in LFP_SESS)}; LFP hours to read: {sum(s.duration for s in LFP_SESS)/3600:.0f} (one block each, plus CA1 blocks)")
print("LS detection channel region labels:", collections.Counter(s.electrodes.region[det_row(s, 'LSRipples')] for s in LFP_SESS))
print("CA1 detection channel region labels:", collections.Counter(s.electrodes.region[det_row(s, 'CA1Ripples')] for s in LFP_SESS if det_row(s, 'CA1Ripples') is not None))

In [ ]:
# Only LS spikes are needed here: the other units' spikes are released to save memory.
for s in S:
    for u in np.flatnonzero(s.unit_region != "ls"): s.spikes[u] = np.zeros(0)
SEED_PL = seed_of("fig1d-phase-locking"); SEED_CA1 = seed_of("ls-spikes-not-locked-to-ca1-ripples"); SEED_COH = seed_of("fig5f-no-phase-coherence")
lock_units = T.lock_units      # per unit: spikes in the epochs, Rayleigh p, resultant length, mean phase, and 10 circular-shift p-values
FS = T.FS_LFP
def nss_peak_times(x, ev):
    """Time and value of the maximum NSS (120-200 Hz, Butterworth order 3) between each event's start and stop."""
    z = T.nss(T.bandpass(x, 120, 200, 3))
    a = np.rint(ev.start_time.to_numpy() * FS).astype(int); b = np.rint(ev.stop_time.to_numpy() * FS).astype(int)
    k = np.array([lo + np.argmax(z[lo:hi + 1]) if hi >= lo else lo for lo, hi in zip(a, b)])
    return k / FS, z[k].astype(np.float64)
def lfp_job(k):
    try:
        return _lfp_job(k)
    except Exception as e:
        return dict(session=LFP_SESS[k].session, subject=LFP_SESS[k].subject, error=repr(e), secs=0)
def _lfp_job(k):
    s = LFP_SESS[k]; t0 = time.time()
    out = dict(session=s.session, subject=s.subject)
    ls_row = det_row(s, "LSRipples"); ca_row = det_row(s, "CA1Ripples") if ca1_ev(s) is not None else None
    ls_units = s.units_in(["ls"]); sg = T.electrode_spike_groups(s.url)
    blk = T.block_rows(s.url, ls_row)
    shank_row = {}
    for sh in np.unique(s.unit_shank[ls_units]):
        cand = [r for r in blk if sg[r] == sh]
        if cand: shank_row[sh] = int(cand[len(cand) // 2])
    X, rows, _ = T.read_lfp_block(s.url, ls_row, [ls_row] + list(shank_row.values()))
    rows = list(rows)
    x = X[:, rows.index(ls_row)].astype(np.float32)
    spk = [s.spikes[u] for u in ls_units]
    rng = np.random.default_rng([SEED_PL, k])
    for band in [(120, 200), (120, 180)]:
        fl = T.bandpass(x, *band, 4); ph = np.angle(T.analytic(fl)).astype(np.float32); m = T.high_power_epochs(fl)
        out[f"lock_{band[1]}"] = lock_units(spk, ph, m, rng)
        if band == (120, 200): out["epoch_frac"] = float(m.mean())
        del fl, ph, m
    # nearest-shank channel, 120-200 Hz
    near = [None] * len(ls_units)
    for sh, r in shank_row.items():
        fl = T.bandpass(X[:, rows.index(r)].astype(np.float32), 120, 200, 4); ph = np.angle(T.analytic(fl)).astype(np.float32); m = T.high_power_epochs(fl)
        sel = np.flatnonzero(s.unit_shank[ls_units] == sh)
        for j, rr in zip(sel, lock_units([spk[j] for j in sel], ph, m, rng)): near[j] = rr
        del fl, ph, m
    out["lock_near"] = near
    del X
    out["ls_nss_t"], out["ls_nss_max"] = nss_peak_times(x, ls_ev(s))
    if ca_row is not None:
        Y, _, _ = T.read_lfp_block(s.url, ca_row, [ca_row]); y = Y[:, 0].astype(np.float32); del Y
        ce = ca1_ev(s)
        out["ca_nss_t"], _ = nss_peak_times(y, ce)
        fl = T.bandpass(y, 120, 200, 4); ph = np.angle(T.analytic(fl)).astype(np.float32)
        rip = np.zeros(len(y), bool)
        for a, b in zip(np.rint(ce.start_time.to_numpy() * FS).astype(int), np.rint(ce.stop_time.to_numpy() * FS).astype(int)): rip[a:b + 1] = True
        rng2 = np.random.default_rng([SEED_CA1, k])
        out["ca1lock_rip"] = lock_units(spk, ph, rip, rng2)
        out["ca1lock_hp"] = lock_units(spk, ph, T.high_power_epochs(fl), rng2)
        del fl, ph, rip
        # coherence, 120-180 Hz: phase-locking value across CA1 ripples of the CA1 minus LS phase, at lags -100..100 ms (2 ms)
        pc = np.angle(T.analytic(T.bandpass(y, 120, 180, 4))).astype(np.float32); pl = np.angle(T.analytic(T.bandpass(x, 120, 180, 4))).astype(np.float32)
        lagsamp = np.rint(np.arange(-100, 101, 2) / 1000 * FS).astype(int)
        def plv(times):
            i = np.rint(times * FS).astype(int)[:, None] + lagsamp[None, :]
            i = i[(i.min(1) >= 0) & (i.max(1) < len(y))]
            return np.abs(np.exp(1j * (pc[i] - pl[i]).astype(np.float64)).mean(0))
        tt = ce.peak_time.to_numpy()
        rng3 = np.random.default_rng([SEED_COH, k])
        out["plv"] = plv(tt); out["plv_jit"] = np.vstack([plv(tt + rng3.uniform(-1, 1, len(tt))) for _ in range(100)])
        out["n_rip"] = len(tt)
    out["secs"] = time.time() - t0
    return out
from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    LFPR = list(ex.map(lfp_job, range(len(LFP_SESS))))
FAILED = [(r["session"], r["error"]) for r in LFPR if "error" in r]
print("Recordings whose job failed:", FAILED if FAILED else "none")
LFPR = [r for r in LFPR if "error" not in r]
print(f"{len(LFPR)} recordings in {(time.time()-t0)/60:.1f} min (per recording {np.median([r['secs'] for r in LFPR]):.0f} s median); "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## fig1d-phase-locking (and dataset-composition, reading a)
> The vast majority of LS neurons (1840 of 1989, 92%) are significantly phase-locked to the 120-200 Hz band (Rayleigh P < 0.05),
> with a strong unimodal preference for the trough of each cycle.

STAR Methods: epochs where the 120-200 Hz power (4th-order Butterworth, 7 ms RMS) is continuously above 2 SD; Hilbert phase of every
spike in them; neurons with more than 50 spikes; Rayleigh p < 0.05; null of 10 circular shifts. Channel readings: the LS detection
channel, and the channel of the neuron's shank (middle site of the shank, where the shank is in the detection channel's storage block).
Band variant 120-180 Hz. Denominators (a) neurons with more than 50 spikes, (b) all LS neurons. Phase: angle of the analytic signal,
trough at +-180 degrees.

In [ ]:
def lock_table(key):
    rows = []
    for r in LFPR:
        for j, u in enumerate(r[key]):
            if u is None: continue
            rows.append(dict(session=r["session"], subject=r["subject"], n=u["n"], p=u["p"], r=u["r"], angle=u["angle"], p_null=u["p_null"]))
    return pd.DataFrame(rows)
PL = {}
for key, nm in [("lock_200", "detection channel, 120-200 Hz"), ("lock_near", "own shank channel, 120-200 Hz"), ("lock_180", "detection channel, 120-180 Hz")]:
    D = lock_table(key); PL[key] = D
    sel = D[D.n > 50]; sig = sel[sel.p < 0.05]
    null = np.nanmean(np.vstack(sel.p_null.values) < 0.05) if len(sel) else np.nan
    mang = np.angle(np.exp(1j * sig.angle).mean()); rt = T.rayleigh(sig.angle.to_numpy())
    print(f"{nm}: LS neurons {len(D)}; with > 50 spikes in high-power epochs {len(sel)}; significant {len(sig)}; fraction (a) {len(sig)/max(len(sel),1):.3f}, "
          f"(b) {len(sig)/max(len(D),1):.3f}; null fraction (10 shifts) {null:.3f}")
    print(f"   preferred phase of significant neurons: circular mean {np.degrees(mang):.1f} deg (|distance to 180| {180-abs(np.degrees(mang)):.1f}); "
          f"Rayleigh over neurons p = {rt[0]:.2g}, r = {rt[1]:.2f}; median resultant length {sig.r.median():.3f}")
D = PL["lock_200"]; sel = D[D.n > 50]
print("\nPer animal (detection channel, 120-200 Hz): neurons > 50 spikes, fraction significant")
print(sel.assign(sig=sel.p < 0.05).groupby("subject").sig.agg(["count", "mean"]).round(3).to_string())
per = sel.assign(sig=sel.p < 0.05).groupby("session").sig.agg(["count", "mean"])
print(f"Neurons with > 50 spikes per recording: {per['count'].min()} to {per['count'].max()}, mean {per['count'].mean():.1f} +- {per['count'].std():.1f} "
      f"(paper: 8-74, 29.6 +- 15.5); recordings {len(per)}")
print(f"dataset-composition reading (a): LS neurons with > 50 spikes in high-power epochs = {len(sel)} (paper 1989, tolerance 198.9)")
print(f"High-power epochs cover a median {np.median([r['epoch_frac'] for r in LFPR]):.3f} of each recording")
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(sel.r, 50); ax[0].set_xlabel("resultant length"); ax[1].hist(np.degrees(sel[sel.p < .05].angle), 36); ax[1].set_xlabel("preferred phase (deg)"); plt.show()
print(f"Resultant length range {sel.r.min():.3f} to {sel.r.max():.3f}")

**Result: dataset-composition partially reproduced, resting on an assumption.** Reading (a): 1996 LS neurons have more than 50 spikes in high-power epochs (paper 1989 +- 198.9), 5 to 62 per recording (mean 27.3 +- 14.7) in 73 recordings. With the counts of the main notebook: 6 animals; CA1 recordings 38 (within); CA3 recordings 10 or 19 (outside 25 +- 2.5); LS HFO rate recordings 77 or 80 (outside 89 +- 8.9); CA1 ripple rate recordings 55 with ripplesALL (within; 47 with CA1Ripples alone); mean LS units per recording 31.1 (within). Four of the six counts other than animals are within tolerance. Two of the four rest on choices the paper does not make, adopted as assumptions: units labelled `hpc` counted as CA1 (the `ca3_channel` flag gives 29 CA1 recordings, outside) and ripplesALL counted as CA1 ripples where CA1Ripples is missing (47 recordings, outside). Without either, 3 of 6 counts are within tolerance and the outcome would be not reproduced.

**Result: fig1d-phase-locking partially reproduced.** Of 1996 LS neurons with more than 50 spikes in high-power epochs, 60.6% are significantly locked to the 120-200 Hz phase of the detection channel (52.0% of all 2327 LS neurons), and 78.2% of 1129 neurons to their own shank's channel where that shank lies in the detection channel's storage block, against 92.5% +- 9.25%; the own-shank notebook adds the other shanks (732 of 863 neurons locked), which brings the own-shank reading to 1615 of 1992 (81.1%), still outside tolerance; the 120-180 Hz band gives 61.3%. Ten circular shifts give 5.2%. The trough preference holds: significant neurons prefer -167.0 degrees (13.0 from the trough), Rayleigh over neurons p < 1e-300. Per animal the fraction ranges from 39.3% (DT9) to 76.8% (DT1).

## ls-spikes-not-locked-to-ca1-ripples
> LS spiking is not phase-locked to the individual waves of hippocampal SPW-Rs.

The same locking analysis with the 120-200 Hz phase of the CA1 detection channel, (a) inside CA1 ripples (CA1Ripples start to stop),
(b) inside the high-power epochs of that channel. One-sided binomial test of the fraction significant against 5%.

In [ ]:
for key, nm in [("ca1lock_rip", "(a) inside CA1 ripples"), ("ca1lock_hp", "(b) CA1 high-power epochs")]:
    rows = [dict(session=r["session"], subject=r["subject"], **{k: u[k] for k in ("n", "p", "r")}, p_null=u["p_null"]) for r in LFPR if key in r for u in r[key]]
    D = pd.DataFrame(rows); sel = D[D.n > 50]; k = int((sel.p < .05).sum())
    bt = stats.binomtest(k, len(sel), 0.05, alternative="greater")
    null = np.nanmean(np.vstack(sel.p_null.values) < 0.05)
    print(f"{nm}: LS neurons {len(D)} in {D.session.nunique()} recordings; > 50 spikes {len(sel)}; significant {k} ({k/max(len(sel),1):.3f}); "
          f"binomial p against 5% = {bt.pvalue:.2g}; null fraction (10 shifts) {null:.3f}; median resultant length {sel.r.median():.3f}")
    print("   per animal:", sel.assign(sig=sel.p < .05).groupby("subject").sig.agg(["count", "mean"]).round(3).to_dict("index"))
sel0 = PL["lock_200"][PL["lock_200"].n > 50]
print(f"Beside fig1d-phase-locking (LS channel): fraction {np.mean(sel0.p < .05):.3f}, median resultant length {sel0.r.median():.3f}")

**Result: ls-spikes-not-locked-to-ca1-ripples not reproduced, resting on an assumption.** Inside CA1 ripples (reading a), 43 of 599 LS neurons with more than 50 spikes (7.2%) are locked to the CA1 channel's 120-200 Hz phase, above 5% (binomial p = 0.012), with a median resultant length of 0.068 (against 0.136 to the LS channel); shifted phases give 4.9%. In the CA1 high-power epochs (reading b), 5.0% (61 of 1218; p = 0.51), at chance. Reading (a), spikes during the SPW-Rs themselves, is adopted as the reading of "individual waves of hippocampal SPW-Rs". The locking found is weak: the fraction is 1.4 times chance.

## fig5d-lag, event times at the maximum NSS
The reading of the event time as the maximum-NSS bin (STAR Methods, "peak power bin"), on the detection channels.

In [ ]:
lg = []
for r in LFPR:
    if "ca_nss_t" not in r: continue
    l, c = np.sort(r["ls_nss_t"]), np.sort(r["ca_nss_t"])
    lg.append(dict(session=r["session"], subject=r["subject"], lag=nearest_lag(l, c) * 1000))
LGN = pd.DataFrame(lg)
for win in [50, 25]:
    per = [np.median(x[np.abs(x) <= win]) for x in LGN.lag if (np.abs(x) <= win).sum()]
    allx = np.concatenate([x[np.abs(x) <= win] for x in LGN.lag])
    print(f"max-NSS times, within +-{win} ms: {len(allx)} pairs in {len(per)} recordings; (a) median of recording medians {np.median(per):.2f} ms; "
          f"(b) pooled median {np.median(allx):.2f} ms; CA1 leading in {np.mean(allx > 0):.3f}")

**Result: fig5d-lag, maximum-NSS reading.** With event times at the maximum NSS of the detection channels, the median lag is 6.00 ms over recording medians and 6.40 ms pooled (6241 pairs within +-50 ms, 36 recordings), CA1 leading in 64.2% of pairs; within +-25 ms, 3.40 and 4.00 ms. This agrees with the deposited-peak reading of the main notebook (6.20 and 6.40 ms).

## fig1h-no-amplitude-frequency-correlation, amplitude from the LFP
> LS HFOs show no significant correlation between event amplitude and peak frequency.

The criterion's amplitude is the maximum NSS of the event. Here it is computed from the LS detection channel (120-200 Hz, Butterworth
order 3, squared and z-scored over the recording; STAR Methods, Ripple detection) between each LSRipples event's start and stop, and
compared with the deposited `peak_normed_power` used in the main notebook. Frequency is the deposited `peak_frequency`. Pearson and Spearman.

In [ ]:
from scipy import stats
rows = []
for r in LFPR:
    s = next(x for x in LFP_SESS if x.session == r["session"]); e = ls_ev(s)
    rows.append(pd.DataFrame(dict(session=r["session"], subject=r["subject"], nss_max=r["ls_nss_max"], peak_normed_power=e.peak_normed_power.to_numpy(),
                                  peak_amplitude=e.peak_amplitude.to_numpy(), peak_frequency=e.peak_frequency.to_numpy())))
AMP = pd.concat(rows)
per = AMP.groupby("session").apply(lambda g: stats.spearmanr(g.nss_max, g.peak_normed_power).statistic if len(g) > 2 else np.nan)
per2 = AMP.groupby("session").apply(lambda g: stats.spearmanr(g.nss_max, g.peak_amplitude).statistic if len(g) > 2 else np.nan)
print(f"{len(AMP)} LS HFOs in {AMP.session.nunique()} recordings. Maximum NSS from the LFP: {AMP.nss_max.min():.2f} to {AMP.nss_max.max():.2f} (median {AMP.nss_max.median():.2f}); "
      f"deposited peak_normed_power {AMP.peak_normed_power.min():.2f} to {AMP.peak_normed_power.max():.2f} (median {AMP.peak_normed_power.median():.2f})")
print(f"Spearman, maximum NSS against peak_normed_power: pooled {stats.spearmanr(AMP.nss_max, AMP.peak_normed_power).statistic:.3f}; per recording median {per.median():.3f} "
      f"(range {per.min():.3f} to {per.max():.3f}). Against peak_amplitude: pooled {stats.spearmanr(AMP.nss_max, AMP.peak_amplitude).statistic:.3f}; per recording median {per2.median():.3f}")
for col in ["nss_max", "peak_normed_power"]:
    pr = stats.pearsonr(AMP[col], AMP.peak_frequency); sr = stats.spearmanr(AMP[col], AMP.peak_frequency)
    print(f"{col} against peak_frequency: Pearson r = {pr.statistic:.3f} (p = {pr.pvalue:.2g}); Spearman rho = {sr.statistic:.3f} (p = {sr.pvalue:.2g})")

**Result: fig1h-no-amplitude-frequency-correlation, amplitude from the LFP.** Over 25690 LS HFOs in 73 recordings, the maximum NSS recomputed from the detection channel (120-200 Hz, order 3) ranks the events much as the deposited `peak_normed_power` does within a recording (median Spearman 0.877, range -0.500 to 0.969) but not across recordings (pooled 0.563); its scale differs (median 4.69 against 8.03), so the lab normalized the signal differently. Against the peak frequency it gives a weak but significant correlation (Pearson r = 0.081, p = 1.3e-38; Spearman rho = 0.116, p = 3.2e-77), while `peak_normed_power` gives none in the same events (Pearson r = 0.005, p = 0.41; Spearman p = 0.11). The outcome in the main notebook rests on the deposited value.

## fig5f-no-phase-coherence
> CA1-LS 120-180 Hz phase coherence around CA1 ripples does not differ from that of randomly jittered windows.

The phase-locking value across CA1 ripples of the CA1 minus LS 120-180 Hz phase, at lags -100..100 ms around the ripple time,
against 100 sets of windows jittered by a uniform +-1 s; averaged over recordings; bounds mean +-3 SD of the jittered sets.

In [ ]:
C = [r for r in LFPR if "plv" in r]
act = np.mean([r["plv"] for r in C], 0); jit = np.mean([r["plv_jit"] for r in C], 0)      # [lags], [100, lags]
lagp = np.arange(-100, 101, 2); z0 = lagp == 0
mu, sd = jit.mean(0), jit.std(0, ddof=1)
print(f"{len(C)} recordings, {sum(r['n_rip'] for r in C)} CA1 ripples")
print(f"At the ripple time: PLV {act[z0][0]:.4f}; jittered mean {mu[z0][0]:.4f}, SD {sd[z0][0]:.4f}; upper bound (+3 SD) {mu[z0][0] + 3*sd[z0][0]:.4f}; "
      f"within bounds: {abs(act[z0][0] - mu[z0][0]) <= 3*sd[z0][0]}")
print(f"Lags (of {len(lagp)}) above +3 SD: {int((act > mu + 3*sd).sum())}; largest excess {np.max((act - mu) / sd):.1f} SD at {lagp[np.argmax((act - mu)/sd)]} ms")
print("Per recording at 0 ms, PLV against its own jittered mean and SD (first 10):",
      [(r["session"][:12], round(float(r["plv"][50]), 3), round(float(r["plv_jit"][:, 50].mean()), 3), round(float(r["plv_jit"][:, 50].std()), 3)) for r in C[:10]])
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(lagp, act, label="ripples"); ax.fill_between(lagp, mu - 3*sd, mu + 3*sd, alpha=.3, label="jittered +-3 SD")
ax.set_xlabel("lag (ms)"); ax.set_ylabel("PLV"); ax.legend(); plt.show()
print(f"Plotted PLV range {act.min():.4f} to {act.max():.4f}")

**Result: fig5f-no-phase-coherence not reproduced.** Over 41 recordings and 91316 CA1 ripples, the CA1-LS 120-180 Hz phase-locking value at the ripple time is 0.186, above the jittered windows' mean of 0.104 by more than 3 SD (SD 0.0072; bound 0.126); 14 of 101 lags lie above the bound, the largest 14.7 SD at 4 ms. In each of the first 10 recordings printed, the value at the ripple time exceeds its own jittered mean.

## fig2-confined-to-ls and discussion-medial-septum (part a)
> HFO events are seen only at recording sites within LS, not at sites in neighbouring structures.

Recordings whose electrodes are labelled in LS and in another structure other than the hippocampus (and not `unknown`). Every
channel is read for the two storage time chunks (about 14 min each) holding the most LSRipples events. Per site: mean 120-180 Hz
power (4th-order Butterworth, squared) within +-25 ms of the LS HFO times in those chunks, divided by its mean at as many random
times in the same chunks (the claim's seed). Sites by the electrodes' region label.

In [ ]:
SITE_SESS = [s for s in S if ls_ev(s) is not None and (s.electrodes.region == "ls").any()
             and len(set(s.electrodes.region) - {"ls", "hpc", "ca1", "ca3", "unknown"}) > 0]
print(f"Recordings with LS and other non-hippocampal sites: {len(SITE_SESS)}: {[(s.session[:22], dict(collections.Counter(s.electrodes.region))) for s in SITE_SESS]}")
def site_job(k):
    s = SITE_SESS[k]; rng = np.random.default_rng([seed_of("fig2-confined-to-ls"), k])
    f = T.open_nwb(s.url); ch = f["processing/ecephys/LFP/lfp/data"].chunks[0]; f.close()
    t = ls_ev(s).peak_time.to_numpy(); kk = np.bincount((t * FS // ch).astype(int))
    best = np.argsort(kk)[::-1][:2]
    pieces, el = T.read_lfp_all(s.url, sorted(best))
    num = np.zeros(len(el)); den = np.zeros(len(el)); nev = 0
    w = int(0.025 * FS)
    for off, blk in pieces:
        tt = (t * FS).astype(int) - off; tt = tt[(tt > w) & (tt < len(blk) - w)]
        if len(tt) == 0: continue
        rt = rng.integers(w, len(blk) - w, len(tt))
        idx = (tt[:, None] + np.arange(-w, w + 1)[None]).ravel(); ridx = (rt[:, None] + np.arange(-w, w + 1)[None]).ravel()
        for c0 in range(0, blk.shape[1], 16):                                   # 16 channels at a time, to bound memory
            P = T.bandpass(blk[:, c0:c0 + 16].astype(np.float32).T, 120, 180, 4) ** 2
            num[c0:c0 + 16] += P[:, idx].sum(1); den[c0:c0 + 16] += P[:, ridx].sum(1)
        nev += len(tt)
    reg = s.electrodes.region.to_numpy()[el]
    # discussion-cingulate-down-state, reading (b): CA1 ripple-triggered LFP of the acg sites in the same chunks
    trig = None
    if (reg == "acg").any() and ca1_ev(s) is not None:
        ct = ca1_ev(s).peak_time.to_numpy(); L = int(0.2 * FS); acc = []; ac = np.flatnonzero(reg == "acg")
        for off, blk in pieces:
            c = (ct * FS).astype(int) - off; c = c[(c > L) & (c < len(blk) - L)]
            if len(c): acc.append(blk[(c[:, None] + np.arange(-L, L + 1)[None]), :][:, :, ac].astype(np.float32).mean(2))
        if acc:
            m = np.vstack(acc).mean(0); trig = (m - m[:int(0.1 * FS)].mean(), sum(len(a) for a in acc))
    del pieces
    sg = T.electrode_spike_groups(s.url)
    return pd.DataFrame(dict(session=s.session[:25], row=el, shank=sg[el], region=reg, ratio=num / den, n_events=nev)), trig
SJ = [site_job(k) for k in range(len(SITE_SESS))]
SITES = pd.concat([a for a, _ in SJ])
tab = SITES.groupby(["session", "region"]).ratio.agg(["count", "median", "max"]).round(2)
print(tab.to_string())
ok_rec = []
for sess, g in SITES.groupby("session"):
    med = g.groupby("region").ratio.median()
    others = [r for r in med.index if r not in ("ls", "hpc", "ca1", "ca3", "unknown")]
    cond = all(med["ls"] > med[r] for r in others) and all((g[g.region == r].ratio < 1.5).all() for r in others)
    ok_rec.append(cond)
    print(f"{sess}: LS median {med['ls']:.2f}; others {[(r, round(med[r], 2), round(g[g.region == r].ratio.max(), 2)) for r in others]}; holds: {cond}; events used {g.n_events.iloc[0]}")
print(f"fig2 criterion holds in {sum(ok_rec)} of {len(ok_rec)} recordings")
# Region labels against channel order (verifier round 1): per recording and shank, the sites in the electrode table's order with their
# label (h hpc, c cc, l ls, A acg, m ms, s sfi) and ratio. A label boundary should be a change in the profile if labels and channels agree.
for sess, g in SITES.groupby("session"):
    print(f"\n{sess}, per shank (site label:ratio in channel order):")
    for sh, gg in g.groupby("shank"):
        print(f"  shank {sh}: " + " ".join(f"{('A' if r == 'acg' else r[0])}:{v:.1f}" for r, v in zip(gg.region, gg.ratio)))
ms = SITES[SITES.region == "ms"]
for sess, g in SITES[SITES.session.isin(ms.session.unique())].groupby("session"):
    print(f"medial septum (a), {sess}: ms site median ratio {g[g.region == 'ms'].ratio.median():.2f} (n {int((g.region == 'ms').sum())}) against LS {g[g.region == 'ls'].ratio.median():.2f}")

**Result: fig2-confined-to-ls not reproduced.** In none of the 8 recordings with LS and other non-hippocampal sites does the criterion hold. LS sites have a higher median 120-180 Hz power ratio at HFO times than the acg sites in all 6 recordings with acg sites (e.g. 3.75 against 0.98), but medial septal sites reach ratios above the LS median in 3 recordings (median 12.32 against 4.97 in 20170320), and in every recording some sites outside LS exceed 1.5 (the largest such ratio per recording is 3.49 to 44.33). The ratios rest on 17 to 113 events in two chunks of each recording. The site profiles show why the labels cannot settle it: on the septal shank (shank 7) the labels ls, ms, acg and cc alternate along the channel order (in 20170320: l, l, l, A, m, l, m, l, m, ...), and sites labelled acg or ms between LS sites have LS-like ratios (16.2, 13.3, 16.4), while the acg sites at the end of the shank are near 1. The dataset stores no site positions, so whether the labels or the channel order are wrong cannot be told; in 20170325 the LS sites' own median ratio is 0.97.

## discussion-cingulate-down-state, reading (b)
The CA1 ripple-triggered LFP averaged over the `acg` sites, in the same time chunks, baseline-subtracted (-200 to -100 ms); the delay is
the time of its largest deviation between 0 and 200 ms.

In [ ]:
tl = np.arange(-int(0.2 * FS), int(0.2 * FS) + 1) / FS * 1000
for (df, tr), s in zip(SJ, SITE_SESS):
    if tr is None: continue
    x, n = tr; w = (tl >= 0) & (tl <= 200)
    k = np.argmax(np.abs(x[w]))
    print(f"{s.session[:25]}: {n} CA1 ripples; extremum of the acg ripple-triggered LFP at {tl[w][k]:.1f} ms ({x[w][k]:.1f} in the file's LFP units); "
          f"minimum at {tl[w][np.argmin(x[w])]:.1f} ms, maximum at {tl[w][np.argmax(x[w])]:.1f} ms")

**Result: discussion-cingulate-down-state not reproduced.** Reading (b): the extremum of the acg ripple-triggered LFP between 0 and 200 ms lies at 2.4 to 7.2 ms in 5 of 6 recordings and at 81.6 ms in one, not at 50 +- 5 ms (one recording has a minimum at 52.0 ms, smaller than its extremum at 2.4 ms). Reading (a), in the main notebook, places the units' minimum at 105 ms.

## Summary

The outcomes of all claims, with how the analysis differs from the paper's and what was found about the dataset, are in the summary of `reproduce_tingley_2019.ipynb`.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.